In [2]:

import os
import json
import re
import unicodedata
import numpy as np
import pandas as pd

# Original dataset location
DATA_DIR = "/kaggle/input/datasets/shwethabodakunt20/humanvsai"

# Save all generated files here
PROJECT_DIR = "/kaggle/working/human-vs-ai-text-detection"
DATA_CLEAN_DIR = os.path.join(PROJECT_DIR, "data", "cleaned")
RESULTS_DIR = os.path.join(PROJECT_DIR, "results")
MODELS_DIR = os.path.join(PROJECT_DIR, "models")

# Create folders
for folder in [DATA_CLEAN_DIR, RESULTS_DIR, MODELS_DIR]:
    os.makedirs(folder, exist_ok=True)

print("Project setup completed!")
print("Dataset exists:", os.path.exists(DATA_DIR))

if os.path.exists(DATA_DIR):
    print("Dataset files:", os.listdir(DATA_DIR))

print("\nProject directory:", PROJECT_DIR)


Project setup completed!
Dataset exists: True
Dataset files: ['archive_filelist.csv', 'eng.train.history.json', 'eng.test.history.json']

Project directory: /kaggle/working/human-vs-ai-text-detection


In [3]:

import json
import os
import re
import unicodedata
import numpy as np
import pandas as pd

# Kaggle paths
DATA_DIR = "/kaggle/input/datasets/shwethabodakunt20/humanvsai"
OUT_DIR = DATA_CLEAN_DIR

SEED = 42
VAL_FRAC = 0.10

# Refusal detection pattern from your original script
REFUSAL = re.compile(
    r"^(?:@\w+\s+)*"
    r"(?:"
    r"(?:I|i)\s*(?:can['’]?t|cannot|can not|won['’]?t|am unable to|['’]?m unable to|"
    r"am not able to|['’]?m not able to)\s+"
    r"(?:fulfill|generate|provide|assist|engage|create|write|comply|continue|"
    r"respond|participate|support|answer|do that|help with|help you|help out|"
    r"produce|complete|process|take part)"
    r"|(?:I|i)['’]?m sorry,? but I"
    r"|(?:Sorry|sorry),? but I (?:can['’]?t|cannot|won['’]?t)"
    r"|(?:I|i) (?:apologize|apologise),? but"
    r"|As an AI"
    r")"
)

def clean_text(t):
    t = unicodedata.normalize("NFC", t)
    t = t.replace("\r\n", "\n").replace("\r", "\n")
    t = re.sub(r"[ \t]+", " ", t)
    return t.strip()

def load(path):
    with open(path, encoding="utf-8") as f:
        data = json.load(f)

    rows = [
        (i, source, clean_text(text))
        for source, texts in data.items()
        for i, text in enumerate(texts)
    ]

    df = pd.DataFrame(rows, columns=["idx", "source", "text"])
    return df[df["text"] != ""].copy()

def flag_cross_class(df):
    counts = df.groupby("text")["source"].nunique()
    df = df.copy()
    df["cross_class_dup"] = df["text"].isin(
        counts[counts > 1].index
    )
    return df

def add_columns(df):
    df = df.copy()
    df["is_refusal"] = df["text"].map(
        lambda t: bool(REFUSAL.match(t))
    )
    df["binary"] = (df["source"] != "human").astype(int)
    df["family"] = df["source"].str.extract(
        r"^(human|qwen3|llama3|mistral7b)", expand=False
    )

    columns = [
        "idx", "source", "binary", "family", "text",
        "cross_class_dup", "is_refusal"
    ]
    return df[columns].reset_index(drop=True)

# Load original data
train = load(f"{DATA_DIR}/eng.train.history.json")
test = load(f"{DATA_DIR}/eng.test.history.json")

# Identify cross-class duplicates
train = flag_cross_class(train)
test = flag_cross_class(test)

# Remove duplicates within each source class
train = train.drop_duplicates(
    subset=["source", "text"], keep="first"
)
test = test.drop_duplicates(
    subset=["source", "text"], keep="first"
)

# Remove test texts that also occur in training
train_texts = set(train["text"])
test = test[~test["text"].isin(train_texts)].copy()

# Add labels and metadata
train = add_columns(train)
test = add_columns(test)

# Reproduce original validation split
rng = np.random.RandomState(SEED)
all_idx = np.array(sorted(train["idx"].unique()))

val_idx = rng.choice(
    all_idx,
    size=int(round(len(all_idx) * VAL_FRAC)),
    replace=False
)

is_val = train["idx"].isin(set(val_idx.tolist()))

val = train[is_val].reset_index(drop=True)
train = train[~is_val].reset_index(drop=True)

# Save cleaned datasets
os.makedirs(OUT_DIR, exist_ok=True)

train.to_csv(f"{OUT_DIR}/train_clean.csv", index=False)
val.to_csv(f"{OUT_DIR}/val_clean.csv", index=False)
test.to_csv(f"{OUT_DIR}/test_clean.csv", index=False)

print("DATA CLEANING COMPLETED")
print("Training rows:", len(train))
print("Validation rows:", len(val))
print("Testing rows:", len(test))
print("\nSaved files:")
print(os.listdir(OUT_DIR))


DATA CLEANING COMPLETED
Training rows: 33948
Validation rows: 3775
Testing rows: 6410

Saved files:
['val_clean.csv', 'test_clean.csv', 'train_clean.csv']


In [4]:

# Load cleaned datasets
train_df = pd.read_csv(f"{DATA_CLEAN_DIR}/train_clean.csv")
val_df = pd.read_csv(f"{DATA_CLEAN_DIR}/val_clean.csv")
test_df = pd.read_csv(f"{DATA_CLEAN_DIR}/test_clean.csv")

# Check class distribution
print("TRAINING CLASS DISTRIBUTION")
print(train_df["source"].value_counts().sort_index())

print("\nVALIDATION CLASS DISTRIBUTION")
print(val_df["source"].value_counts().sort_index())

print("\nTEST CLASS DISTRIBUTION")
print(test_df["source"].value_counts().sort_index())

# Check missing values
print("\nMISSING TEXT VALUES")
print("Train:", train_df["text"].isna().sum())
print("Validation:", val_df["text"].isna().sum())
print("Test:", test_df["text"].isna().sum())

# Verify number of classes
print("\nNumber of classes:", train_df["source"].nunique())
print("Class names:", sorted(train_df["source"].unique()))


TRAINING CLASS DISTRIBUTION
source
human           3419
llama3_1b       3278
llama3_3b       3368
llama3_8b       3409
mistral7b_v1    3388
mistral7b_v2    3420
mistral7b_v3    3419
qwen3_14b       3416
qwen3_4b        3415
qwen3_8b        3416
Name: count, dtype: int64

VALIDATION CLASS DISTRIBUTION
source
human           380
llama3_1b       366
llama3_3b       375
llama3_8b       380
mistral7b_v1    377
mistral7b_v2    380
mistral7b_v3    380
qwen3_14b       379
qwen3_4b        379
qwen3_8b        379
Name: count, dtype: int64

TEST CLASS DISTRIBUTION
source
human           648
llama3_1b       606
llama3_3b       633
llama3_8b       646
mistral7b_v1    637
mistral7b_v2    649
mistral7b_v3    648
qwen3_14b       649
qwen3_4b        647
qwen3_8b        647
Name: count, dtype: int64

MISSING TEXT VALUES
Train: 0
Validation: 0
Test: 0

Number of classes: 10
Class names: ['human', 'llama3_1b', 'llama3_3b', 'llama3_8b', 'mistral7b_v1', 'mistral7b_v2', 'mistral7b_v3', 'qwen3_14b', 'qwen3_4b

In [11]:

import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):",
          round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    print("No GPU detected. Please enable GPU in Kaggle Notebook Settings.")


PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
GPU memory (GB): 14.56


In [14]:
import torch
import transformers
import sklearn
import joblib

print("All libraries imported successfully!")
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("GPU available:", torch.cuda.is_available())

All libraries imported successfully!
PyTorch: 2.11.0+cu128
Transformers: 5.16.1
GPU available: True


In [16]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "ai-forever/mGPT"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16
)

model = model.to("cuda")
model.eval()

print("mGPT model loaded successfully!")
print("Model:", MODEL_NAME)
print("Device:", next(model.parameters()).device)
print("Data type:", next(model.parameters()).dtype)

vocab.json:   0%|          | 0.00/1.89M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.20M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/606 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 3.45GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: ai-forever/mGPT
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...23}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.45GB            

model.safetensors: downloading bytes:           |  0.00B            

mGPT model loaded successfully!
Model: ai-forever/mGPT
Device: cuda:0
Data type: torch.float16


In [17]:
import torch

sample_text = "Artificial intelligence is transforming the world."

inputs = tokenizer(
    sample_text,
    return_tensors="pt"
).to("cuda")

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits[:, :-1, :].float()

probabilities = torch.softmax(logits, dim=-1)

token_entropy = -(
    probabilities * torch.log(probabilities.clamp(min=1e-12))
).sum(dim=-1)

mean_entropy = token_entropy.mean().item()

print("Sample text:", sample_text)
print("Number of tokens:", inputs["input_ids"].shape[1])
print("Token-level entropy:", token_entropy.cpu().numpy())
print("Mean entropy:", mean_entropy)

Sample text: Artificial intelligence is transforming the world.
Number of tokens: 10
Token-level entropy: [[3.9269757 0.5685952 4.905715  5.336586  4.5626903 0.4992463 3.6985788
  3.8285174 3.3448658]]
Mean entropy: 3.4079747200012207


In [18]:
train_df = pd.read_csv(os.path.join(DATA_CLEAN_DIR, "train_clean.csv"))
val_df = pd.read_csv(os.path.join(DATA_CLEAN_DIR, "val_clean.csv"))
test_df = pd.read_csv(os.path.join(DATA_CLEAN_DIR, "test_clean.csv"))

print("Training dataset:", train_df.shape)
print("Validation dataset:", val_df.shape)
print("Testing dataset:", test_df.shape)

print("\nRequired columns:")
print(train_df[["source", "binary", "text"]].head())

print("\nMissing text values:")
print("Train:", train_df["text"].isna().sum())
print("Validation:", val_df["text"].isna().sum())
print("Test:", test_df["text"].isna().sum())

Training dataset: (33948, 7)
Validation dataset: (3775, 7)
Testing dataset: (6410, 7)

Required columns:
  source  binary                                               text
0  human       0            @nypost She has a water hose she’s fine
1  human       0             @NemoPays There’s a reason they’re VIP
2  human       0  @simonateba @realDonaldTrump The worse part of...
3  human       0  @RepMTG She's smart enough to know that you do...
4  human       0               @Zee207702 Good morning sweetheart 🌸

Missing text values:
Train: 0
Validation: 0
Test: 0


In [19]:
import numpy as np
import torch

def calculate_entropy(texts):
    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    ).to("cuda")

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits[:, :-1, :].float()
    probabilities = torch.softmax(logits, dim=-1)

    token_entropy = -(
        probabilities * torch.log(probabilities.clamp(min=1e-12))
    ).sum(dim=-1)

    valid_tokens = inputs["attention_mask"][:, 1:]

    mean_entropy = (
        (token_entropy * valid_tokens).sum(dim=1)
        / valid_tokens.sum(dim=1).clamp(min=1)
    )

    return mean_entropy.cpu().numpy()


sample_texts = train_df["text"].head(5).tolist()

sample_entropies = calculate_entropy(sample_texts)

for i, entropy in enumerate(sample_entropies):
    print(f"Sample {i+1}: Entropy = {entropy:.4f}")

Sample 1: Entropy = 3.8391
Sample 2: Entropy = 4.2603
Sample 3: Entropy = 3.9893
Sample 4: Entropy = 4.0672
Sample 5: Entropy = 4.2141


In [20]:
from tqdm.auto import tqdm

BATCH_SIZE = 4
CHECKPOINT_EVERY = 100

train_entropy_path = os.path.join(
    RESULTS_DIR, "train_entropy.csv"
)

# Start a fresh training entropy file
if os.path.exists(train_entropy_path):
    os.remove(train_entropy_path)

results = []

for start in tqdm(range(0, len(train_df), BATCH_SIZE)):
    batch = train_df.iloc[start:start + BATCH_SIZE]

    entropy_values = calculate_entropy(batch["text"].tolist())

    for position, entropy in zip(batch.index, entropy_values):
        row = train_df.loc[position]

        results.append({
            "row_index": position,
            "source": row["source"],
            "binary": row["binary"],
            "entropy": float(entropy)
        })

    # Save a checkpoint every 100 batches
    if (start // BATCH_SIZE + 1) % CHECKPOINT_EVERY == 0:
        pd.DataFrame(results).to_csv(
            train_entropy_path,
            mode="a",
            header=not os.path.exists(train_entropy_path),
            index=False
        )
        results = []

# Save remaining results
if results:
    pd.DataFrame(results).to_csv(
        train_entropy_path,
        mode="a",
        header=not os.path.exists(train_entropy_path),
        index=False
    )

print("\nTraining entropy calculation completed!")
print("Saved to:", train_entropy_path)

  0%|          | 0/8487 [00:00<?, ?it/s]


Training entropy calculation completed!
Saved to: /kaggle/working/human-vs-ai-text-detection/results/train_entropy.csv


In [21]:
train_entropy_df = pd.read_csv(train_entropy_path)

print("Saved rows:", len(train_entropy_df))
print("Expected rows:", len(train_df))
print("Missing entropy values:", train_entropy_df["entropy"].isna().sum())
print("Duplicate row indices:", train_entropy_df["row_index"].duplicated().sum())

print("\nEntropy statistics:")
print(train_entropy_df["entropy"].describe())

print("\nEntropy by class:")
print(
    train_entropy_df.groupby("source")["entropy"]
    .agg(["count", "mean", "std"])
    .round(4)
)

Saved rows: 33948
Expected rows: 33948
Missing entropy values: 0
Duplicate row indices: 0

Entropy statistics:
count    33948.000000
mean         3.847494
std          0.747856
min          0.071273
25%          3.413826
50%          3.822925
75%          4.262401
max          7.698652
Name: entropy, dtype: float64

Entropy by class:
              count    mean     std
source                             
human          3419  3.7535  0.6538
llama3_1b      3278  3.9962  0.8211
llama3_3b      3368  3.7690  0.7101
llama3_8b      3409  3.8268  0.6708
mistral7b_v1   3388  3.7866  0.8434
mistral7b_v2   3420  4.2362  1.0622
mistral7b_v3   3419  3.8147  0.6923
qwen3_14b      3416  3.7539  0.5716
qwen3_4b       3415  3.7672  0.5511
qwen3_8b       3416  3.7748  0.6169


In [ ]:
val_entropy_path = os.path.join(
    RESULTS_DIR, "val_entropy.csv"
)

val_results = []

for start in tqdm(range(0, len(val_df), BATCH_SIZE)):
    batch = val_df.iloc[start:start + BATCH_SIZE]

    entropy_values = calculate_entropy(batch["text"].tolist())

    for position, entropy in zip(batch.index, entropy_values):
        row = val_df.loc[position]

        val_results.append({
            "row_index": position,
            "source": row["source"],
            "binary": row["binary"],
            "entropy": float(entropy)
        })

pd.DataFrame(val_results).to_csv(
    val_entropy_path, index=False
)

print("Validation entropy calculation completed!")
print("Rows saved:", len(val_results))
print("Saved to:", val_entropy_path)

  0%|          | 0/944 [00:00<?, ?it/s]

In [25]:
val_entropy_df = pd.read_csv(val_entropy_path)

print("Saved rows:", len(val_entropy_df))
print("Expected rows:", len(val_df))
print("Missing entropy values:", val_entropy_df["entropy"].isna().sum())
print("Duplicate row indices:", val_entropy_df["row_index"].duplicated().sum())

print("\nValidation entropy statistics:")
print(val_entropy_df["entropy"].describe())

print("\nEntropy by class:")
print(
    val_entropy_df.groupby("source")["entropy"]
    ["entropy"].agg(["count", "mean", "std"]).round(4)
)

Saved rows: 3775
Expected rows: 3775
Missing entropy values: 0
Duplicate row indices: 0

Validation entropy statistics:
count    3775.000000
mean        3.843262
std         0.765864
min         0.135869
25%         3.402199
50%         3.807131
75%         4.264813
max         7.406258
Name: entropy, dtype: float64

Entropy by class:


IndexError: Column(s) entropy already selected

In [24]:
print("Validation entropy by class:")

print(
    val_entropy_df.groupby("source")["entropy"]
    .agg(["count", "mean", "std"])
    .round(4)
)

Validation entropy by class:
              count    mean     std
source                             
human           380  3.7374  0.6634
llama3_1b       366  4.0482  0.8780
llama3_3b       375  3.6934  0.6844
llama3_8b       380  3.7548  0.6718
mistral7b_v1    377  3.7449  0.7493
mistral7b_v2    380  4.3223  1.1413
mistral7b_v3    380  3.8587  0.7881
qwen3_14b       379  3.7774  0.5521
qwen3_4b        379  3.7482  0.5358
qwen3_8b        379  3.7515  0.5752


In [26]:
test_entropy_path = os.path.join(
    RESULTS_DIR, "test_entropy.csv"
)

test_results = []

for start in tqdm(range(0, len(test_df), BATCH_SIZE)):
    batch = test_df.iloc[start:start + BATCH_SIZE]

    entropy_values = calculate_entropy(batch["text"].tolist())

    for position, entropy in zip(batch.index, entropy_values):
        row = test_df.loc[position]

        test_results.append({
            "row_index": position,
            "source": row["source"],
            "binary": row["binary"],
            "entropy": float(entropy)
        })

    # Save progress every 100 batches
    if (start // BATCH_SIZE + 1) % 100 == 0:
        pd.DataFrame(test_results).to_csv(
            test_entropy_path,
            mode="a",
            header=not os.path.exists(test_entropy_path),
            index=False
        )
        test_results = []

if test_results:
    pd.DataFrame(test_results).to_csv(
        test_entropy_path,
        mode="a",
        header=not os.path.exists(test_entropy_path),
        index=False
    )

print("\nTest entropy calculation completed!")
print("Saved to:", test_entropy_path)

  0%|          | 0/1603 [00:00<?, ?it/s]


Test entropy calculation completed!
Saved to: /kaggle/working/human-vs-ai-text-detection/results/test_entropy.csv


In [28]:
train_entropy_df = pd.read_csv(train_entropy_path)
val_entropy_df = pd.read_csv(val_entropy_path)
test_entropy_df = pd.read_csv(test_entropy_path)

datasets = {
    "Training": train_entropy_df,
    "Validation": val_entropy_df,
    "Testing": test_entropy_df
}

for name, df in datasets.items():
    print(f"\n{name}")
    print("-" * 35)
    print("Rows:", len(df))
    print("Missing values:", df["entropy"].isna().sum())
    print("Duplicate indices:", df["row_index"].duplicated().sum())
    print("Mean entropy:", round(df["entropy"].mean(), 4))
    print("Standard deviation:", round(df["entropy"].std(), 4))

print("\nAll entropy files verified!")


Training
-----------------------------------
Rows: 33948
Missing values: 0
Duplicate indices: 0
Mean entropy: 3.8475
Standard deviation: 0.7479

Validation
-----------------------------------
Rows: 3775
Missing values: 0
Duplicate indices: 0
Mean entropy: 3.8433
Standard deviation: 0.7659

Testing
-----------------------------------
Rows: 6410
Missing values: 0
Duplicate indices: 0
Mean entropy: 3.8507
Standard deviation: 0.7214

All entropy files verified!


In [29]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    balanced_accuracy_score
)

# Binary classification features and labels
X_train_binary = train_entropy_df[["entropy"]]
y_train_binary = train_entropy_df["binary"]

X_val_binary = val_entropy_df[["entropy"]]
y_val_binary = val_entropy_df["binary"]

X_test_binary = test_entropy_df[["entropy"]]
y_test_binary = test_entropy_df["binary"]

# Multiclass classification features and labels
X_train_multi = train_entropy_df[["entropy"]]
y_train_multi = train_entropy_df["source"]

X_val_multi = val_entropy_df[["entropy"]]
y_val_multi = val_entropy_df["source"]

X_test_multi = test_entropy_df[["entropy"]]
y_test_multi = test_entropy_df["source"]

print("Binary classification:")
print("Training:", X_train_binary.shape)
print("Validation:", X_val_binary.shape)
print("Testing:", X_test_binary.shape)
print("\nTraining label distribution:")
print(y_train_binary.value_counts().sort_index())

print("\nMulticlass classification:")
print("Number of classes:", y_train_multi.nunique())
print("Classes:", sorted(y_train_multi.unique()))

Binary classification:
Training: (33948, 1)
Validation: (3775, 1)
Testing: (6410, 1)

Training label distribution:
binary
0     3419
1    30529
Name: count, dtype: int64

Multiclass classification:
Number of classes: 10
Classes: ['human', 'llama3_1b', 'llama3_3b', 'llama3_8b', 'mistral7b_v1', 'mistral7b_v2', 'mistral7b_v3', 'qwen3_14b', 'qwen3_4b', 'qwen3_8b']


In [30]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

binary_rf = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

# Train using training data only
binary_rf.fit(X_train_binary, y_train_binary)

# Evaluate on validation data
val_binary_predictions = binary_rf.predict(X_val_binary)

print("BINARY RANDOM FOREST - VALIDATION RESULTS")
print("=" * 50)

print("\nAccuracy:",
      round(accuracy_score(y_val_binary, val_binary_predictions), 4))

print("Balanced accuracy:",
      round(balanced_accuracy_score(y_val_binary, val_binary_predictions), 4))

print("Macro F1:",
      round(f1_score(y_val_binary, val_binary_predictions, average="macro"), 4))

print("\nClassification Report:")
print(classification_report(
    y_val_binary,
    val_binary_predictions,
    target_names=["Human", "AI"],
    zero_division=0
))

print("Confusion Matrix:")
print(confusion_matrix(y_val_binary, val_binary_predictions))

BINARY RANDOM FOREST - VALIDATION RESULTS

Accuracy: 0.8207
Balanced accuracy: 0.5123
Macro F1: 0.5121

Classification Report:
              precision    recall  f1-score   support

       Human       0.12      0.13      0.12       380
          AI       0.90      0.90      0.90      3395

    accuracy                           0.82      3775
   macro avg       0.51      0.51      0.51      3775
weighted avg       0.82      0.82      0.82      3775

Confusion Matrix:
[[  48  332]
 [ 345 3050]]


In [32]:
multi_rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

# Train the 10-class model
multi_rf.fit(X_train_multi, y_train_multi)

# Predict validation data
val_multi_predictions = multi_rf.predict(X_val_multi)

print("10-CLASS RANDOM FOREST - VALIDATION RESULTS")
print("=" * 55)

print("\nAccuracy:",
      round(accuracy_score(y_val_multi, val_multi_predictions), 4))

print("Macro F1:",
      round(f1_score(
          y_val_multi,
          val_multi_predictions,
          average="macro"
      ), 4))

print("\nClassification Report:")
print(classification_report(
    y_val_multi,
    val_multi_predictions,
    labels=sorted(y_train_multi.unique()),
    zero_division=0
))

print("\nConfusion Matrix:")
print(confusion_matrix(
    y_val_multi,
    val_multi_predictions,
    labels=sorted(y_train_multi.unique())
))

10-CLASS RANDOM FOREST - VALIDATION RESULTS

Accuracy: 0.115
Macro F1: 0.1146

Classification Report:
              precision    recall  f1-score   support

       human       0.12      0.12      0.12       380
   llama3_1b       0.07      0.07      0.07       366
   llama3_3b       0.14      0.13      0.14       375
   llama3_8b       0.10      0.10      0.10       380
mistral7b_v1       0.09      0.08      0.09       377
mistral7b_v2       0.23      0.24      0.23       380
mistral7b_v3       0.09      0.09      0.09       380
   qwen3_14b       0.09      0.10      0.10       379
    qwen3_4b       0.10      0.09      0.10       379
    qwen3_8b       0.11      0.11      0.11       379

    accuracy                           0.11      3775
   macro avg       0.11      0.11      0.11      3775
weighted avg       0.11      0.11      0.11      3775


Confusion Matrix:
[[47 37 36 33 38 34 42 45 41 27]
 [40 27 35 36 30 47 35 43 36 37]
 [39 34 50 42 23 35 25 48 41 38]
 [46 42 36 37 28 33 3

In [33]:
from sklearn.model_selection import RandomizedSearchCV
import joblib

multi_param_distributions = {
    "n_estimators": [50, 100, 150, 300],
    "criterion": ["gini", "entropy"],
    "max_features": ["sqrt", "log2", None],
    "max_depth": [None, 10, 100],
    "min_samples_split": [2, 4, 6],
    "min_samples_leaf": [1, 3],
    "bootstrap": [True, False]
}

multi_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(
        random_state=42,
        n_jobs=-1
    ),
    param_distributions=multi_param_distributions,
    n_iter=20,
    scoring="f1_macro",
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

multi_search.fit(X_train_multi, y_train_multi)

best_multi_rf = multi_search.best_estimator_

print("\nBest Cross-Validation Macro F1:",
      round(multi_search.best_score_, 4))

print("\nBest Parameters:")
print(multi_search.best_params_)

joblib.dump(
    best_multi_rf,
    os.path.join(MODELS_DIR, "multiclass_rf_tuned.joblib")
)

print("\nTuned 10-class model saved successfully!")

Fitting 3 folds for each of 20 candidates, totalling 60 fits

Best Cross-Validation Macro F1: 0.1268

Best Parameters:
{'n_estimators': 300, 'min_samples_split': 6, 'min_samples_leaf': 3, 'max_features': 'sqrt', 'max_depth': None, 'criterion': 'gini', 'bootstrap': True}

Tuned 10-class model saved successfully!


In [34]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)

val_multi_tuned_predictions = best_multi_rf.predict(X_val_multi)

print("TUNED 10-CLASS RANDOM FOREST - VALIDATION")
print("=" * 60)

print("\nAccuracy:",
      round(accuracy_score(y_val_multi, val_multi_tuned_predictions), 4))

print("Macro F1:",
      round(f1_score(
          y_val_multi,
          val_multi_tuned_predictions,
          average="macro"
      ), 4))

print("Weighted F1:",
      round(f1_score(
          y_val_multi,
          val_multi_tuned_predictions,
          average="weighted"
      ), 4))

print("\nClassification Report:")

print(classification_report(
    y_val_multi,
    val_multi_tuned_predictions,
    labels=sorted(y_train_multi.unique()),
    zero_division=0
))

print("\nConfusion Matrix:")

print(confusion_matrix(
    y_val_multi,
    val_multi_tuned_predictions,
    labels=sorted(y_train_multi.unique())
))

TUNED 10-CLASS RANDOM FOREST - VALIDATION

Accuracy: 0.1258
Macro F1: 0.1227
Weighted F1: 0.1228

Classification Report:
              precision    recall  f1-score   support

       human       0.13      0.12      0.13       380
   llama3_1b       0.12      0.12      0.12       366
   llama3_3b       0.12      0.12      0.12       375
   llama3_8b       0.10      0.09      0.10       380
mistral7b_v1       0.08      0.07      0.08       377
mistral7b_v2       0.29      0.36      0.32       380
mistral7b_v3       0.11      0.10      0.11       380
   qwen3_14b       0.09      0.09      0.09       379
    qwen3_4b       0.08      0.09      0.09       379
    qwen3_8b       0.09      0.08      0.08       379

    accuracy                           0.13      3775
   macro avg       0.12      0.13      0.12      3775
weighted avg       0.12      0.13      0.12      3775


Confusion Matrix:
[[ 47  38  37  39  42  30  30  42  40  35]
 [ 37  44  39  29  29  50  40  44  30  24]
 [ 33  37  45  

In [35]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)

test_multi_predictions = best_multi_rf.predict(X_test_multi)

test_accuracy = accuracy_score(
    y_test_multi, test_multi_predictions
)
test_macro_f1 = f1_score(
    y_test_multi, test_multi_predictions, average="macro"
)
test_weighted_f1 = f1_score(
    y_test_multi, test_multi_predictions, average="weighted"
)

print("TUNED 10-CLASS RANDOM FOREST - TEST RESULTS")
print("=" * 55)

print(f"Accuracy:     {test_accuracy:.4f}")
print(f"Macro F1:     {test_macro_f1:.4f}")
print(f"Weighted F1:  {test_weighted_f1:.4f}")

print("\nClassification Report:")
print(classification_report(
    y_test_multi,
    test_multi_predictions,
    labels=sorted(y_train_multi.unique()),
    zero_division=0
))

print("\nConfusion Matrix:")
print(confusion_matrix(
    y_test_multi,
    test_multi_predictions,
    labels=sorted(y_train_multi.unique())
))

TUNED 10-CLASS RANDOM FOREST - TEST RESULTS
Accuracy:     0.1296
Macro F1:     0.1283
Weighted F1:  0.1286

Classification Report:
              precision    recall  f1-score   support

       human       0.11      0.11      0.11       648
   llama3_1b       0.11      0.11      0.11       606
   llama3_3b       0.10      0.10      0.10       633
   llama3_8b       0.10      0.09      0.09       646
mistral7b_v1       0.11      0.11      0.11       637
mistral7b_v2       0.27      0.31      0.29       649
mistral7b_v3       0.11      0.11      0.11       648
   qwen3_14b       0.11      0.12      0.12       649
    qwen3_4b       0.10      0.11      0.11       647
    qwen3_8b       0.14      0.13      0.13       647

    accuracy                           0.13      6410
   macro avg       0.13      0.13      0.13      6410
weighted avg       0.13      0.13      0.13      6410


Confusion Matrix:
[[ 70  63  65  57  63  74  61  70  72  53]
 [ 64  64  61  52  50  80  56  68  55  56]
 [ 60

In [38]:
import json
import os
import pandas as pd

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    balanced_accuracy_score
)

classes = sorted(y_train_multi.unique())

# Calculate balanced accuracy
test_balanced_accuracy = balanced_accuracy_score(
    y_test_multi,
    test_multi_predictions
)

results_summary = {
    "experiment": "mGPT Entropy + Random Forest",
    "task": "10-class classification",
    "feature": "Mean token-level predictive entropy",
    "model": "Random Forest",
    "best_parameters": multi_search.best_params_,
    "validation": {
        "accuracy": 0.1258,
        "macro_f1": 0.1227,
        "weighted_f1": 0.1228
    },
    "test": {
        "accuracy": float(test_accuracy),
        "macro_f1": float(test_macro_f1),
        "weighted_f1": float(test_weighted_f1),
        "balanced_accuracy": float(test_balanced_accuracy),
        "classification_report": classification_report(
            y_test_multi,
            test_multi_predictions,
            labels=classes,
            output_dict=True,
            zero_division=0
        )
    }
}

summary_path = os.path.join(
    RESULTS_DIR, "multiclass_rf_results.json"
)

with open(summary_path, "w") as f:
    json.dump(results_summary, f, indent=4)

confusion_df = pd.DataFrame(
    confusion_matrix(
        y_test_multi,
        test_multi_predictions,
        labels=classes
    ),
    index=classes,
    columns=classes
)

confusion_df.to_csv(
    os.path.join(RESULTS_DIR, "multiclass_confusion_matrix.csv")
)

print("Research results saved successfully!")
print("Balanced accuracy:", round(test_balanced_accuracy, 4))
print("JSON:", summary_path)
print("Confusion matrix saved!")

Research results saved successfully!
Balanced accuracy: 0.1293
JSON: /kaggle/working/human-vs-ai-text-detection/results/multiclass_rf_results.json
Confusion matrix saved!


In [41]:
import os

files_to_check = [
    os.path.join(RESULTS_DIR, "multiclass_rf_results.json"),
    os.path.join(RESULTS_DIR, "multiclass_confusion_matrix.csv"),
    os.path.join(MODELS_DIR, "multiclass_rf_tuned.joblib")
]

for file_path in files_to_check:
    print(
        "FOUND:" if os.path.exists(file_path) else "MISSING:",
        file_path
    )

FOUND: /kaggle/working/human-vs-ai-text-detection/results/multiclass_rf_results.json
FOUND: /kaggle/working/human-vs-ai-text-detection/results/multiclass_confusion_matrix.csv
FOUND: /kaggle/working/human-vs-ai-text-detection/models/multiclass_rf_tuned.joblib


In [42]:
import os
import json
import shutil
import zipfile

EXPORT_DIR = "/kaggle/working/github_export"
os.makedirs(EXPORT_DIR, exist_ok=True)

# Copy experiment results
shutil.copy(
    os.path.join(RESULTS_DIR, "multiclass_rf_results.json"),
    EXPORT_DIR
)

shutil.copy(
    os.path.join(RESULTS_DIR, "multiclass_confusion_matrix.csv"),
    EXPORT_DIR
)

# Copy trained model
shutil.copy(
    os.path.join(MODELS_DIR, "multiclass_rf_tuned.joblib"),
    EXPORT_DIR
)

# Create README
readme = """
# Human vs AI-Generated Text Detection

## Experiment: mGPT Entropy + Random Forest

### Objective
10-class classification of human-written and AI-generated text.

### Method
1. Use ai-forever/mGPT to calculate token-level predictive entropy.
2. Average token-level entropy for each text.
3. Train a Random Forest classifier using the mean entropy feature.
4. Evaluate the model on a held-out test set.

### Test Results

- Accuracy: 12.96%
- Macro F1: 12.83%
- Weighted F1: 12.86%
- Balanced Accuracy: 12.93%

### Important Note
This experiment applies an entropy-based method to a 10-class
classification task. The results indicate that mean entropy alone
has limited ability to distinguish the ten classes.

### Files
- multiclass_rf_results.json: Experiment configuration and metrics.
- multiclass_confusion_matrix.csv: Test confusion matrix.
- multiclass_rf_tuned.joblib: Trained Random Forest model.

The original dataset and text samples are not included.
"""

with open(os.path.join(EXPORT_DIR, "README.md"), "w") as f:
    f.write(readme)

# Create .gitignore
gitignore = """
data/raw/
data/cleaned/
*.csv
*.jsonl
.env
*.token
__pycache__/
.ipynb_checkpoints/
"""

# Allow the aggregate result files to be committed
gitignore += """
!multiclass_rf_results.json
!multiclass_confusion_matrix.csv
"""

with open(os.path.join(EXPORT_DIR, ".gitignore"), "w") as f:
    f.write(gitignore)

# Create ZIP
ZIP_PATH = "/kaggle/working/human_vs_ai_github_package.zip"

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zipf:
    for filename in os.listdir(EXPORT_DIR):
        file_path = os.path.join(EXPORT_DIR, filename)
        zipf.write(file_path, arcname=filename)

print("GitHub package created successfully!")
print("Package:", ZIP_PATH)
print("\nIncluded files:")
for filename in os.listdir(EXPORT_DIR):
    print("-", filename)

print("\nZIP size:", round(os.path.getsize(ZIP_PATH) / (1024 * 1024), 2), "MB")

GitHub package created successfully!
Package: /kaggle/working/human_vs_ai_github_package.zip

Included files:
- README.md
- .gitignore
- multiclass_confusion_matrix.csv
- multiclass_rf_tuned.joblib
- multiclass_rf_results.json

ZIP size: 86.88 MB


In [44]:
import os
import subprocess

# Configure Git identity
subprocess.run(
    ["git", "config", "--global", "user.name", "Shwetha Bodakunti"],
    check=True
)

subprocess.run(
    ["git", "config", "--global", "user.email", "YOUR_GITHUB_EMAIL"],
    check=True
)

# Read GitHub token from Kaggle Secrets
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
GITHUB_TOKEN = user_secrets.get_secret("GITHUB_TOKEN")

# Repository details
GITHUB_USERNAME = "shwetha-dev"
REPO_NAME = "human-vs-ai-text-detection"

# Configure authenticated repository URL
REPO_URL = (
    f"https://{GITHUB_USERNAME}:{GITHUB_TOKEN}"
    f"@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
)

print("Git configured successfully!")
print("GitHub token loaded securely.")
print("Repository:", f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}")

Git configured successfully!
GitHub token loaded securely.
Repository: https://github.com/shwetha-dev/human-vs-ai-text-detection


In [47]:
import os
import shutil

PROJECT_DIR = "/kaggle/working/human-vs-ai-text-detection"
EXPORT_DIR = "/kaggle/working/github_export"

# Copy README and .gitignore into the main project
for filename in ["README.md", ".gitignore"]:
    source = os.path.join(EXPORT_DIR, filename)
    destination = os.path.join(PROJECT_DIR, filename)

    shutil.copy2(source, destination)

# Verify required files
required_files = [
    "README.md",
    ".gitignore",
    "results/multiclass_rf_results.json",
    "results/multiclass_confusion_matrix.csv",
    "models/multiclass_rf_tuned.joblib"
]

print("Checking project files:\n")

for filename in required_files:
    path = os.path.join(PROJECT_DIR, filename)
    print(
        "FOUND:" if os.path.exists(path) else "MISSING:",
        filename
    )

Checking project files:

FOUND: README.md
FOUND: .gitignore
FOUND: results/multiclass_rf_results.json
FOUND: results/multiclass_confusion_matrix.csv
FOUND: models/multiclass_rf_tuned.joblib


In [48]:
import subprocess
import os

PROJECT_DIR = "/kaggle/working/human-vs-ai-text-detection"

def git_run(command):
    result = subprocess.run(
        command,
        cwd=PROJECT_DIR,
        capture_output=True,
        text=True
    )
    if result.returncode != 0:
        raise RuntimeError(result.stderr)
    return result.stdout

# Stage the required files
git_run([
    "git", "add",
    "README.md",
    ".gitignore",
    "results/multiclass_rf_results.json",
    "results/multiclass_confusion_matrix.csv",
    "models/multiclass_rf_tuned.joblib"
])

# Check staged files
print(git_run(["git", "status", "--short"]))

# Commit
git_run([
    "git", "commit",
    "-m",
    "Add mGPT entropy Random Forest experiment"
])

# Push to GitHub
print(git_run(["git", "push", "-u", "origin", "main"]))

print("Successfully pushed project to GitHub!")

A  .gitignore
A  README.md
A  models/multiclass_rf_tuned.joblib
A  results/multiclass_confusion_matrix.csv
A  results/multiclass_rf_results.json



RuntimeError: fatal: 'origin' does not appear to be a git repository
fatal: Could not read from remote repository.

Please make sure you have the correct access rights
and the repository exists.


In [50]:
import subprocess
import os

PROJECT_DIR = "/kaggle/working/human-vs-ai-text-detection"

# Reuse the token loaded from Kaggle Secrets
REPO_URL = (
    f"https://shwetha-dev:{GITHUB_TOKEN}"
    "@github.com/shwetha-dev/human-vs-ai-text-detection.git"
)

# Add or update the origin remote
check = subprocess.run(
    ["git", "remote", "get-url", "origin"],
    cwd=PROJECT_DIR,
    capture_output=True,
    text=True
)

if check.returncode == 0:
    subprocess.run(
        ["git", "remote", "set-url", "origin", REPO_URL],
        cwd=PROJECT_DIR,
        check=True
    )
else:
    subprocess.run(
        ["git", "remote", "add", "origin", REPO_URL],
        cwd=PROJECT_DIR,
        check=True
    )

print("GitHub remote configured successfully!")
print("Remote name: origin")
print("Repository: https://github.com/shwetha-dev/human-vs-ai-text-detection")

GitHub remote configured successfully!
Remote name: origin
Repository: https://github.com/shwetha-dev/human-vs-ai-text-detection


In [51]:
import subprocess

result = subprocess.run(
    ["git", "ls-remote", "--heads", "origin"],
    cwd=PROJECT_DIR,
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print("GitHub connection successful!")
    print(result.stdout if result.stdout else "Repository is empty.")
else:
    print("Connection failed:")
    print(result.stderr)

GitHub connection successful!
Repository is empty.


In [53]:
import subprocess

result = subprocess.run(
    ["git", "push", "-u", "origin", "main"],
    cwd=PROJECT_DIR,
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print(result.stdout)
    print("Successfully uploaded your research project!")
    print("Repository:")
    print("https://github.com/shwetha-dev/human-vs-ai-text-detection")
else:
    print("Push failed:")
    print(result.stderr)

Push failed:
remote: error: Trace: 784b8d2a393ad0159464b9234a698a17ecc18159e45eff7726eee8eacc8f8c43        
remote: error: See https://gh.io/lfs for more information.        
remote: error: File models/multiclass_rf_tuned.joblib is 493.56 MB; this exceeds GitHub's file size limit of 100.00 MB        
remote: error: GH001: Large files detected. You may want to try Git Large File Storage - https://git-lfs.github.com.        
To https://github.com/shwetha-dev/human-vs-ai-text-detection.git
 ! [remote rejected] main -> main (pre-receive hook declined)
error: failed to push some refs to 'https://github.com/shwetha-dev/human-vs-ai-text-detection.git'



In [54]:
import subprocess
import os

PROJECT_DIR = "/kaggle/working/human-vs-ai-text-detection"

def run_git(command):
    result = subprocess.run(
        command,
        cwd=PROJECT_DIR,
        capture_output=True,
        text=True
    )
    if result.returncode != 0:
        raise RuntimeError(result.stderr)
    return result.stdout

# Remove model from Git tracking, but keep it in Kaggle
run_git([
    "git", "rm", "--cached",
    "models/multiclass_rf_tuned.joblib"
])

# Add models folder to .gitignore
gitignore_path = os.path.join(PROJECT_DIR, ".gitignore")

with open(gitignore_path, "a") as f:
    f.write("\n# Large trained model files\nmodels/\n")

# Stage updated .gitignore
run_git(["git", "add", ".gitignore"])

# Replace the previous commit so the large file is excluded
run_git([
    "git", "commit", "--amend",
    "--no-edit"
])

print("Large model excluded from Git history.")
print("Model still exists in Kaggle:",
      os.path.exists(os.path.join(
          PROJECT_DIR, "models", "multiclass_rf_tuned.joblib"
      )))
print("\nFiles prepared for GitHub:")
print(run_git(["git", "status", "--short"]))

Large model excluded from Git history.
Model still exists in Kaggle: True

Files prepared for GitHub:



In [55]:
import subprocess

result = subprocess.run(
    ["git", "push", "-u", "origin", "main"],
    cwd=PROJECT_DIR,
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print(result.stdout)
    print("Successfully uploaded your research project!")
    print("https://github.com/shwetha-dev/human-vs-ai-text-detection")
else:
    print("Push failed:")
    print(result.stderr)

branch 'main' set up to track 'origin/main'.

Successfully uploaded your research project!
https://github.com/shwetha-dev/human-vs-ai-text-detection


In [58]:
import os
import json
import pandas as pd

from reportlab.lib import colors
from reportlab.lib.pagesizes import A4, landscape
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer,
    Table, TableStyle, PageBreak, Preformatted
)

PROJECT_DIR = "/kaggle/working/human-vs-ai-text-detection"
RESULTS_DIR = os.path.join(PROJECT_DIR, "results")

PDF_PATH = "/kaggle/working/Human_vs_AI_Research_Report.pdf"

# Load experiment results
with open(os.path.join(RESULTS_DIR, "multiclass_rf_results.json")) as f:
    results = json.load(f)

confusion = pd.read_csv(
    os.path.join(RESULTS_DIR, "multiclass_confusion_matrix.csv"),
    index_col=0
)

styles = getSampleStyleSheet()
story = []

story.append(Paragraph(
    "Human vs AI-Generated Text Detection",
    styles["Title"]
))
story.append(Paragraph(
    "mGPT Entropy + Random Forest — 10-Class Experiment",
    styles["Heading2"]
))
story.append(Spacer(1, 20))

# README
story.append(Paragraph("1. Research Overview", styles["Heading1"]))

readme_path = os.path.join(PROJECT_DIR, "README.md")

if os.path.exists(readme_path):
    with open(readme_path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line.startswith("#"):
                story.append(Paragraph(
                    line.lstrip("# ").replace("&", "&amp;"),
                    styles["Heading2"]
                ))
            elif line:
                story.append(Paragraph(
                    line.replace("&", "&amp;"),
                    styles["BodyText"]
                ))
            story.append(Spacer(1, 5))

# Methodology
story.append(Spacer(1, 15))
story.append(Paragraph("2. Experiment Configuration", styles["Heading1"]))

config = [
    ["Parameter", "Value"],
    ["Experiment", results["experiment"]],
    ["Task", results["task"]],
    ["Feature", results["feature"]],
    ["Classifier", results["model"]],
    ["Best hyperparameters", json.dumps(
        results["best_parameters"], indent=2
    )]
]

table = Table(config, colWidths=[5*cm, 20*cm])
table.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (-1, 0), colors.lightgrey),
    ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
    ("VALIGN", (0, 0), (-1, -1), "TOP"),
    ("FONTSIZE", (0, 0), (-1, -1), 8),
    ("BOTTOMPADDING", (0, 0), (-1, -1), 8),
]))
story.append(table)

# Metrics
story.append(Spacer(1, 20))
story.append(Paragraph("3. Validation and Test Results", styles["Heading1"]))

metric_data = [["Metric", "Validation", "Test"]]

for metric in ["accuracy", "macro_f1", "weighted_f1"]:
    metric_data.append([
        metric.replace("_", " ").title(),
        f'{results["validation"][metric]*100:.2f}%',
        f'{results["test"][metric]*100:.2f}%'
    ])

metric_data.append([
    "Balanced Accuracy",
    "Not recorded",
    f'{results["test"]["balanced_accuracy"]*100:.2f}%'
])

table = Table(metric_data, colWidths=[7*cm, 7*cm, 7*cm])
table.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (-1, 0), colors.lightgrey),
    ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
    ("ALIGN", (1, 1), (-1, -1), "CENTER"),
    ("FONTSIZE", (0, 0), (-1, -1), 9),
    ("BOTTOMPADDING", (0, 0), (-1, -1), 8),
]))
story.append(table)

# Classification report
story.append(PageBreak())
story.append(Paragraph(
    "4. Per-Class Classification Report",
    styles["Heading1"]
))

report = results["test"]["classification_report"]

report_data = [[
    "Class", "Precision", "Recall", "F1-score", "Support"
]]

for class_name, values in report.items():
    if isinstance(values, dict) and "precision" in values:
        report_data.append([
            class_name,
            f'{values["precision"]:.4f}',
            f'{values["recall"]:.4f}',
            f'{values["f1-score"]:.4f}',
            str(values["support"])
        ])

table = Table(report_data, repeatRows=1)
table.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (-1, 0), colors.lightgrey),
    ("GRID", (0, 0), (-1, -1), 0.4, colors.grey),
    ("ALIGN", (1, 1), (-1, -1), "CENTER"),
    ("FONTSIZE", (0, 0), (-1, -1), 8),
    ("BOTTOMPADDING", (0, 0), (-1, -1), 7),
]))
story.append(table)

# Confusion matrix
story.append(PageBreak())
story.append(Paragraph(
    "5. Test Confusion Matrix",
    styles["Heading1"]
))

matrix_data = [["Actual / Predicted"] + list(confusion.columns)]

for index, row in confusion.iterrows():
    matrix_data.append([str(index)] + [
        str(int(value)) for value in row
    ])

table = Table(
    matrix_data,
    repeatRows=1,
    colWidths=[3.5*cm] + [1.8*cm]*len(confusion.columns)
)

table.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (-1, 0), colors.lightgrey),
    ("GRID", (0, 0), (-1, -1), 0.4, colors.grey),
    ("ALIGN", (1, 1), (-1, -1), "CENTER"),
    ("FONTSIZE", (0, 0), (-1, -1), 7),
    ("BOTTOMPADDING", (0, 0), (-1, -1), 7),
]))
story.append(table)

# Project file inventory
story.append(PageBreak())
story.append(Paragraph(
    "6. Project Files and Artifacts",
    styles["Heading1"]
))

file_data = [["File", "Description", "Size (MB)"]]

project_files = [
    ("README.md", "Research overview and methodology"),
    (".gitignore", "Git exclusion rules"),
    ("results/multiclass_rf_results.json", "Experiment metrics"),
    ("results/multiclass_confusion_matrix.csv", "Confusion matrix"),
    ("models/multiclass_rf_tuned.joblib", "Trained Random Forest model")
]

for filename, description in project_files:
    path = os.path.join(PROJECT_DIR, filename)
    size = (
        round(os.path.getsize(path) / (1024*1024), 2)
        if os.path.exists(path) else "Missing"
    )
    file_data.append([filename, description, str(size)])

table = Table(file_data, colWidths=[9*cm, 10*cm, 3*cm])
table.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (-1, 0), colors.lightgrey),
    ("GRID", (0, 0), (-1, -1), 0.4, colors.grey),
    ("FONTSIZE", (0, 0), (-1, -1), 8),
    ("VALIGN", (0, 0), (-1, -1), "TOP"),
]))
story.append(table)

story.append(Spacer(1, 15))
story.append(Paragraph(
    "Note: The trained model is a separate binary artifact. "
    "It is not embedded in this PDF.",
    styles["BodyText"]
))

# Generate PDF
doc = SimpleDocTemplate(
    PDF_PATH,
    pagesize=landscape(A4),
    rightMargin=1.5*cm,
    leftMargin=1.5*cm,
    topMargin=1.5*cm,
    bottomMargin=1.5*cm
)

doc.build(story)

print("PDF generated successfully!")
print("Location:", PDF_PATH)
print("Size:", round(os.path.getsize(PDF_PATH)/(1024*1024), 2), "MB")

PDF generated successfully!
Location: /kaggle/working/Human_vs_AI_Research_Report.pdf
Size: 0.01 MB


In [57]:
!pip install reportlab -q

import reportlab
print("ReportLab installed successfully!")
print("Version:", reportlab.Version)

ReportLab installed successfully!
Version: 5.1.0
